# NYC 311 2025 — Data Profiling

This notebook profiles the validated 2025 NYC 311 analytical dataset to understand its structure, service demand patterns, resolution performance, and data quality characteristics.

**Analysis period:** January–December 2025


In [2]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

parquet_path = "/content/drive/MyDrive/NYC_311_2025_BACKUP/NYC_311_2025_FINAL.parquet"

df = pd.read_parquet(parquet_path)

print("DATASET LOADED")
print("=" * 70)
print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))
print("Memory usage:", f"{df.memory_usage(deep=True).sum() / (1024**3):.2f} GB")


Mounted at /content/drive
DATASET LOADED
Rows: 3,655,040
Columns: 19
Memory usage: 3.31 GB


## 1. Dataset Structure



In [3]:
# DATA PROFILING — DATASET STRUCTURE

print("DATASET STRUCTURE PROFILE")
print("=" * 80)

print("\n1. SHAPE")
print("-" * 80)
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")

print("\n2. COLUMN INFORMATION")
print("-" * 80)

profile = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "non_null": df.notna().sum().values,
    "missing": df.isna().sum().values,
    "missing_pct": (
        df.isna().mean().values * 100
    ).round(2)
})

print(profile.to_string(index=False))

print("\n3. DUPLICATE CHECK")
print("-" * 80)

duplicate_rows = df.duplicated().sum()
duplicate_ids = df["service_request_id"].duplicated().sum()

print("Duplicate complete rows:", f"{duplicate_rows:,}")
print("Duplicate service request IDs:", f"{duplicate_ids:,}")

print("\n" + "=" * 80)
print("STRUCTURE PROFILE COMPLETE")
print("=" * 80)


DATASET STRUCTURE PROFILE

1. SHAPE
--------------------------------------------------------------------------------
Rows: 3,655,040
Columns: 19

2. COLUMN INFORMATION
--------------------------------------------------------------------------------
                 column          dtype  non_null  missing  missing_pct
     service_request_id          int64   3655040        0         0.00
             created_at datetime64[ns]   3655040        0         0.00
              closed_at datetime64[ns]   3526004   129036         3.53
       resolution_hours        float64   3525111   129929         3.55
        resolution_days        float64   3525111   129929         3.55
     resolution_quality         object   3655040        0         0.00
                 agency         object   3655040        0         0.00
         complaint_type         string   3652868     2172         0.06
             descriptor         object   3572341    82699         2.26
                 status         string   

## 2. Categorical Distributions



In [4]:
# DATA PROFILING — CATEGORICAL DISTRIBUTIONS

print("CATEGORICAL DISTRIBUTION PROFILE")
print("=" * 80)

# 1. AGENCY DISTRIBUTION

print("\n1. SERVICE REQUESTS BY AGENCY")
print("-" * 80)

agency_profile = (
    df["agency"]
    .value_counts()
    .rename_axis("agency")
    .reset_index(name="request_count")
)

agency_profile["share_pct"] = (
    agency_profile["request_count"]
    / len(df)
    * 100
).round(2)

print(agency_profile.to_string(index=False))

# 2. COMPLAINT TYPE DISTRIBUTION

print("\n2. TOP 20 COMPLAINT TYPES")
print("-" * 80)

complaint_profile = (
    df["complaint_type"]
    .value_counts(dropna=False)
    .head(20)
    .rename_axis("complaint_type")
    .reset_index(name="request_count")
)

complaint_profile["share_pct"] = (
    complaint_profile["request_count"]
    / len(df)
    * 100
).round(2)

print(complaint_profile.to_string(index=False))

# 3. BOROUGH DISTRIBUTION

print("\n3. REQUESTS BY BOROUGH")
print("-" * 80)

borough_profile = (
    df["borough"]
    .value_counts(dropna=False)
    .rename_axis("borough")
    .reset_index(name="request_count")
)

borough_profile["share_pct"] = (
    borough_profile["request_count"]
    / len(df)
    * 100
).round(2)

print(borough_profile.to_string(index=False))

# 4. STATUS DISTRIBUTION

print("\n4. REQUESTS BY STATUS")
print("-" * 80)

status_profile = (
    df["status"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="request_count")
)

status_profile["share_pct"] = (
    status_profile["request_count"]
    / len(df)
    * 100
).round(2)

print(status_profile.to_string(index=False))

print("\n" + "=" * 80)
print("CATEGORICAL DISTRIBUTION PROFILE COMPLETE")
print("=" * 80)


CATEGORICAL DISTRIBUTION PROFILE

1. SERVICE REQUESTS BY AGENCY
--------------------------------------------------------------------------------
                                            agency  request_count  share_pct
                   New York City Police Department        1717594      46.99
Department of Housing Preservation and Development         774535      21.19
                          Department of Sanitation         326780       8.94
                      Department of Transportation         212103       5.80
            Department of Environmental Protection         197143       5.39
                Department of Parks and Recreation         110855       3.03
                           Department of Buildings         103066       2.82
           Department of Health and Mental Hygiene          83396       2.28
                   Department of Homeless Services          51449       1.41
                     Taxi and Limousine Commission          36225       0.99
        

## 3. Monthly Service Demand



In [5]:
# DATA PROFILING — MONTHLY REQUEST VOLUME

print("MONTHLY REQUEST VOLUME PROFILE")
print("=" * 80)

# Create month fields for profiling
df["created_month"] = df["created_at"].dt.month
df["created_month_name"] = df["created_at"].dt.strftime("%B")

monthly_profile = (
    df.groupby(
        ["created_month", "created_month_name"],
        as_index=False
    )
    .size()
    .rename(columns={"size": "request_count"})
    .sort_values("created_month")
)

monthly_profile["share_pct"] = (
    monthly_profile["request_count"]
    / len(df)
    * 100
).round(2)

print("\nMONTHLY REQUEST VOLUME")
print("-" * 80)

print(
    monthly_profile[
        [
            "created_month",
            "created_month_name",
            "request_count",
            "share_pct"
        ]
    ].to_string(index=False)
)

print("\nTOTAL REQUESTS:", f"{monthly_profile['request_count'].sum():,}")
print(
    "MONTHS COVERED:",
    monthly_profile["created_month"].nunique()
)

print("\n" + "=" * 80)
print("MONTHLY REQUEST VOLUME PROFILE COMPLETE")
print("=" * 80)


MONTHLY REQUEST VOLUME PROFILE

MONTHLY REQUEST VOLUME
--------------------------------------------------------------------------------
 created_month created_month_name  request_count  share_pct
             1            January         348180       9.53
             2           February         255364       6.99
             3              March         281221       7.69
             4              April         272549       7.46
             5                May         295058       8.07
             6               June         306442       8.38
             7               July         315883       8.64
             8             August         304039       8.32
             9          September         302684       8.28
            10            October         336612       9.21
            11           November         304905       8.34
            12           December         332103       9.09

TOTAL REQUESTS: 3,655,040
MONTHS COVERED: 12

MONTHLY REQUEST VOLUME PROFILE COMPLE

## 4. Resolution Time Profile



In [6]:
# DATA PROFILING — RESOLUTION TIME

print("RESOLUTION TIME PROFILE")
print("=" * 80)

resolution_stats = df["resolution_hours"].describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
)

print("\nRESOLUTION TIME — HOURS")
print("-" * 80)
print(resolution_stats)

print("\nRESOLUTION TIME — DAYS")
print("-" * 80)

resolution_days_stats = df["resolution_days"].describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
)

print(resolution_days_stats)

print("\nVALID RESOLUTION RECORDS")
print("-" * 80)

valid_resolution = df["resolution_hours"].notna().sum()
total_records = len(df)

print("Valid resolution records:", f"{valid_resolution:,}")
print("Missing/invalid resolution:", f"{total_records - valid_resolution:,}")
print(
    "Resolution coverage:",
    f"{valid_resolution / total_records * 100:.2f}%"
)

print("\n" + "=" * 80)
print("RESOLUTION TIME PROFILE COMPLETE")
print("=" * 80)


RESOLUTION TIME PROFILE

RESOLUTION TIME — HOURS
--------------------------------------------------------------------------------
count    3.525111e+06
mean     2.346127e+02
std      8.843923e+02
min      0.000000e+00
50%      6.683333e+00
75%      6.735556e+01
90%      4.082333e+02
95%      1.233460e+03
99%      4.879780e+03
max      1.498477e+04
Name: resolution_hours, dtype: float64

RESOLUTION TIME — DAYS
--------------------------------------------------------------------------------
count    3.525111e+06
mean     9.775531e+00
std      3.684968e+01
min      0.000000e+00
50%      2.784722e-01
75%      2.806481e+00
90%      1.700972e+01
95%      5.139415e+01
99%      2.033242e+02
max      6.243653e+02
Name: resolution_days, dtype: float64

VALID RESOLUTION RECORDS
--------------------------------------------------------------------------------
Valid resolution records: 3,525,111
Missing/invalid resolution: 129,929
Resolution coverage: 96.45%

RESOLUTION TIME PROFILE COMPLETE


## 5. Resolution Time Bands



In [7]:
# DATA PROFILING — RESOLUTION TIME BANDS

print("RESOLUTION TIME BAND PROFILE")
print("=" * 80)

df["resolution_band"] = pd.cut(
    df["resolution_hours"],
    bins=[
        -0.001,
        1,
        6,
        24,
        72,
        168,
        720,
        float("inf")
    ],
    labels=[
        "0–1 hour",
        "1–6 hours",
        "6–24 hours",
        "1–3 days",
        "3–7 days",
        "7–30 days",
        "30+ days"
    ],
    include_lowest=True
)

resolution_band_profile = (
    df["resolution_band"]
    .value_counts(sort=False, dropna=False)
    .rename_axis("resolution_band")
    .reset_index(name="request_count")
)

resolution_band_profile["share_pct"] = (
    resolution_band_profile["request_count"]
    / df["resolution_band"].notna().sum()
    * 100
).round(2)

print("\nRESOLUTION BANDS")
print("-" * 80)

print(
    resolution_band_profile.to_string(index=False)
)

print("\nVALID RESOLUTION RECORDS:",
      f"{df['resolution_band'].notna().sum():,}")

print("\n" + "=" * 80)
print("RESOLUTION TIME BAND PROFILE COMPLETE")
print("=" * 80)


RESOLUTION TIME BAND PROFILE

RESOLUTION BANDS
--------------------------------------------------------------------------------
resolution_band  request_count  share_pct
       0–1 hour         855716      24.27
      1–6 hours         871170      24.71
     6–24 hours         465196      13.20
       1–3 days         488961      13.87
       3–7 days         283488       8.04
      7–30 days         304436       8.64
       30+ days         256144       7.27
            NaN         129929       3.69

VALID RESOLUTION RECORDS: 3,525,111

RESOLUTION TIME BAND PROFILE COMPLETE


## 6. Agency Resolution Profile



In [8]:
# DATA PROFILING — AGENCY RESOLUTION PROFILE

print("AGENCY RESOLUTION PROFILE")
print("=" * 80)

agency_resolution_profile = (
    df.groupby("agency")
    .agg(
        request_count=("service_request_id", "size"),
        valid_resolution_count=("resolution_hours", "count"),
        median_resolution_hours=("resolution_hours", "median"),
        p75_resolution_hours=(
            "resolution_hours",
            lambda x: x.quantile(0.75)
        ),
        p90_resolution_hours=(
            "resolution_hours",
            lambda x: x.quantile(0.90)
        )
    )
    .reset_index()
)

agency_resolution_profile["resolution_coverage_pct"] = (
    agency_resolution_profile["valid_resolution_count"]
    / agency_resolution_profile["request_count"]
    * 100
).round(2)

agency_resolution_profile["median_resolution_days"] = (
    agency_resolution_profile["median_resolution_hours"]
    / 24
).round(2)

agency_resolution_profile = (
    agency_resolution_profile
    .sort_values("request_count", ascending=False)
)

print(
    agency_resolution_profile[
        [
            "agency",
            "request_count",
            "valid_resolution_count",
            "resolution_coverage_pct",
            "median_resolution_hours",
            "median_resolution_days",
            "p75_resolution_hours",
            "p90_resolution_hours"
        ]
    ].to_string(index=False)
)

print("\n" + "=" * 80)
print("AGENCY RESOLUTION PROFILE COMPLETE")
print("=" * 80)


AGENCY RESOLUTION PROFILE
                                            agency  request_count  valid_resolution_count  resolution_coverage_pct  median_resolution_hours  median_resolution_days  p75_resolution_hours  p90_resolution_hours
                   New York City Police Department        1717594                 1681009                    97.87                 1.269167                    0.05              3.099722              6.807500
Department of Housing Preservation and Development         774535                  764540                    98.71                78.595833                    3.27            317.936806            956.066111
                          Department of Sanitation         326780                  315127                    96.43                32.988056                    1.37             77.582222            174.292667
                      Department of Transportation         212103                  200339                    94.45                46.333333   

## 7. Complaint Types by Agency



In [9]:
# DATA PROFILING — TOP COMPLAINT TYPES BY AGENCY

print("TOP COMPLAINT TYPES BY AGENCY")
print("=" * 80)

agency_complaint_profile = (
    df.dropna(subset=["complaint_type"])
      .groupby(
          ["agency", "complaint_type"],
          as_index=False
      )
      .size()
      .rename(columns={"size": "request_count"})
)

agency_complaint_profile["agency_share_pct"] = (
    agency_complaint_profile["request_count"]
    / agency_complaint_profile.groupby("agency")["request_count"]
      .transform("sum")
    * 100
).round(2)

agency_complaint_profile["rank_within_agency"] = (
    agency_complaint_profile
    .groupby("agency")["request_count"]
    .rank(
        method="first",
        ascending=False
    )
)

top5_agency_complaints = (
    agency_complaint_profile[
        agency_complaint_profile["rank_within_agency"] <= 5
    ]
    .sort_values(
        ["agency", "request_count"],
        ascending=[True, False]
    )
)

print(
    top5_agency_complaints[
        [
            "agency",
            "complaint_type",
            "request_count",
            "agency_share_pct",
            "rank_within_agency"
        ]
    ].to_string(index=False)
)

print("\n" + "=" * 80)
print("TOP COMPLAINT TYPES BY AGENCY PROFILE COMPLETE")
print("=" * 80)


TOP COMPLAINT TYPES BY AGENCY
                                            agency                     complaint_type  request_count  agency_share_pct  rank_within_agency
                           Department of Buildings      General Construction/Plumbing          34123             33.11                 1.0
                           Department of Buildings                           Elevator          20912             20.29                 2.0
                           Department of Buildings                       Building/Use          20087             19.49                 3.0
                           Department of Buildings              Real Time Enforcement           5501              5.34                 4.0
                           Department of Buildings                            Boilers           3939              3.82                 5.0
      Department of Consumer and Worker Protection                 Consumer Complaint          17810             97.44                 1

## 8. Borough Demand and Resolution



In [10]:
# DATA PROFILING — BOROUGH SERVICE DEMAND & RESOLUTION

print("BOROUGH SERVICE DEMAND & RESOLUTION PROFILE")
print("=" * 80)

borough_profile = (
    df.groupby("borough", dropna=False)
      .agg(
          request_count=("service_request_id", "size"),
          valid_resolution_count=("resolution_hours", "count"),
          median_resolution_hours=("resolution_hours", "median"),
          p75_resolution_hours=(
              "resolution_hours",
              lambda x: x.quantile(0.75)
          ),
          p90_resolution_hours=(
              "resolution_hours",
              lambda x: x.quantile(0.90)
          )
      )
      .reset_index()
)

borough_profile["request_share_pct"] = (
    borough_profile["request_count"]
    / borough_profile["request_count"].sum()
    * 100
).round(2)

borough_profile["resolution_coverage_pct"] = (
    borough_profile["valid_resolution_count"]
    / borough_profile["request_count"]
    * 100
).round(2)

borough_profile["median_resolution_days"] = (
    borough_profile["median_resolution_hours"]
    / 24
).round(2)

borough_profile = borough_profile.sort_values(
    "request_count",
    ascending=False
)

print(
    borough_profile[
        [
            "borough",
            "request_count",
            "request_share_pct",
            "valid_resolution_count",
            "resolution_coverage_pct",
            "median_resolution_hours",
            "median_resolution_days",
            "p75_resolution_hours",
            "p90_resolution_hours"
        ]
    ].to_string(index=False)
)

print("\n" + "=" * 80)
print("BOROUGH SERVICE DEMAND & RESOLUTION PROFILE COMPLETE")
print("=" * 80)


BOROUGH SERVICE DEMAND & RESOLUTION PROFILE
      borough  request_count  request_share_pct  valid_resolution_count  resolution_coverage_pct  median_resolution_hours  median_resolution_days  p75_resolution_hours  p90_resolution_hours
     BROOKLYN        1003412              27.45                  974009                    97.07                 6.315278                    0.26             71.366667            378.780833
       QUEENS         813632              22.26                  784212                    96.38                 4.163611                    0.17             45.826944            330.545000
        BRONX         781563              21.38                  768854                    98.37                 9.741667                    0.41             61.653889            357.270139
    MANHATTAN         673197              18.42                  646985                    96.11                11.050000                    0.46             87.100000            671.003333
      

## 9. Community Board Demand and Resolution



In [11]:
# DATA PROFILING — COMMUNITY BOARD DEMAND & RESOLUTION

print("COMMUNITY BOARD DEMAND & RESOLUTION PROFILE")
print("=" * 80)

community_board_profile = (
    df.dropna(subset=["community_board"])
      .groupby("community_board")
      .agg(
          request_count=("service_request_id", "size"),
          valid_resolution_count=("resolution_hours", "count"),
          median_resolution_hours=("resolution_hours", "median"),
          p75_resolution_hours=(
              "resolution_hours",
              lambda x: x.quantile(0.75)
          ),
          p90_resolution_hours=(
              "resolution_hours",
              lambda x: x.quantile(0.90)
          )
      )
      .reset_index()
)

community_board_profile["request_share_pct"] = (
    community_board_profile["request_count"]
    / len(df)
    * 100
).round(3)

community_board_profile["resolution_coverage_pct"] = (
    community_board_profile["valid_resolution_count"]
    / community_board_profile["request_count"]
    * 100
).round(2)

community_board_profile["median_resolution_days"] = (
    community_board_profile["median_resolution_hours"]
    / 24
).round(2)

community_board_profile = community_board_profile.sort_values(
    "request_count",
    ascending=False
)

print("TOP 15 COMMUNITY BOARDS BY REQUEST VOLUME")
print("-" * 80)

print(
    community_board_profile.head(15)[
        [
            "community_board",
            "request_count",
            "request_share_pct",
            "resolution_coverage_pct",
            "median_resolution_hours",
            "median_resolution_days",
            "p75_resolution_hours",
            "p90_resolution_hours"
        ]
    ].to_string(index=False)
)

print("\n" + "=" * 80)
print("COMMUNITY BOARD PROFILE SUMMARY")
print("=" * 80)

print("Total validated community boards:", len(community_board_profile))
print(
    "Requests represented:",
    f"{community_board_profile['request_count'].sum():,}"
)
print(
    "Coverage of full dataset:",
    f"{community_board_profile['request_count'].sum() / len(df) * 100:.2f}%"
)

print("\n" + "=" * 80)
print("COMMUNITY BOARD DEMAND & RESOLUTION PROFILE COMPLETE")
print("=" * 80)


COMMUNITY BOARD DEMAND & RESOLUTION PROFILE
TOP 15 COMMUNITY BOARDS BY REQUEST VOLUME
--------------------------------------------------------------------------------
community_board  request_count  request_share_pct  resolution_coverage_pct  median_resolution_hours  median_resolution_days  p75_resolution_hours  p90_resolution_hours
       12 BRONX         182831              5.002                    99.55                 3.583333                    0.15             15.765833             72.952500
   12 MANHATTAN         101122              2.767                    98.70                15.033333                    0.63             76.891667            406.828611
       04 BRONX          80797              2.211                    98.34                24.578889                    1.02            115.375417            483.253333
    01 BROOKLYN          80680              2.207                    98.24                 3.676944                    0.15             69.719306            470.

## 10. Monthly Resolution Performance



In [12]:
# DATA PROFILING — MONTHLY RESOLUTION PROFILE

print("MONTHLY RESOLUTION PROFILE")
print("=" * 80)

monthly_resolution_profile = (
    df.groupby(["created_month", "created_month_name"])
      .agg(
          request_count=("service_request_id", "size"),
          valid_resolution_count=("resolution_hours", "count"),
          median_resolution_hours=("resolution_hours", "median"),
          p75_resolution_hours=(
              "resolution_hours",
              lambda x: x.quantile(0.75)
          ),
          p90_resolution_hours=(
              "resolution_hours",
              lambda x: x.quantile(0.90)
          )
      )
      .reset_index()
      .sort_values("created_month")
)

monthly_resolution_profile["resolution_coverage_pct"] = (
    monthly_resolution_profile["valid_resolution_count"]
    / monthly_resolution_profile["request_count"]
    * 100
).round(2)

monthly_resolution_profile["median_resolution_days"] = (
    monthly_resolution_profile["median_resolution_hours"]
    / 24
).round(2)

monthly_resolution_profile["request_share_pct"] = (
    monthly_resolution_profile["request_count"]
    / monthly_resolution_profile["request_count"].sum()
    * 100
).round(2)

print(
    monthly_resolution_profile[
        [
            "created_month_name",
            "request_count",
            "request_share_pct",
            "resolution_coverage_pct",
            "median_resolution_hours",
            "median_resolution_days",
            "p75_resolution_hours",
            "p90_resolution_hours"
        ]
    ].to_string(index=False)
)

print("\n" + "=" * 80)
print("MONTHLY RESOLUTION PROFILE COMPLETE")
print("=" * 80)


MONTHLY RESOLUTION PROFILE
created_month_name  request_count  request_share_pct  resolution_coverage_pct  median_resolution_hours  median_resolution_days  p75_resolution_hours  p90_resolution_hours
           January         348180               9.53                    99.22                15.089583                    0.63             68.526042            394.418556
          February         255364               6.99                    98.56                13.793056                    0.57             71.083542            445.878500
             March         281221               7.69                    98.54                 8.415000                    0.35             57.265833            478.125556
             April         272549               7.46                    98.16                 6.984306                    0.29             67.801736            453.118611
               May         295058               8.07                    98.48                 4.138056                

## 11. Monthly × Agency Resolution Coverage



In [13]:
# DATA QUALITY — MONTH × AGENCY RESOLUTION COVERAGE

print("MONTH × AGENCY RESOLUTION COVERAGE")
print("=" * 80)

month_agency_resolution = (
    df.groupby(
        ["created_month_name", "created_month", "agency"]
    )
    .agg(
        request_count=("service_request_id", "size"),
        valid_resolution_count=("resolution_hours", "count")
    )
    .reset_index()
)

month_agency_resolution["resolution_coverage_pct"] = (
    month_agency_resolution["valid_resolution_count"]
    / month_agency_resolution["request_count"]
    * 100
).round(2)

# Focus on June and August
anomaly_months = month_agency_resolution[
    month_agency_resolution["created_month"].isin([6, 8])
].copy()

anomaly_months = anomaly_months.sort_values(
    ["created_month", "request_count"],
    ascending=[True, False]
)

print(
    anomaly_months[
        [
            "created_month_name",
            "agency",
            "request_count",
            "valid_resolution_count",
            "resolution_coverage_pct"
        ]
    ].to_string(index=False)
)

print("\n" + "=" * 80)
print("MONTH × AGENCY RESOLUTION COVERAGE CHECK COMPLETE")
print("=" * 80)


MONTH × AGENCY RESOLUTION COVERAGE
created_month_name                                             agency  request_count  valid_resolution_count  resolution_coverage_pct
              June                    New York City Police Department         148225                  123910                    83.60
              June Department of Housing Preservation and Development          40906                   37202                    90.95
              June                           Department of Sanitation          29657                   23498                    79.23
              June             Department of Environmental Protection          24422                   21280                    87.13
              June                       Department of Transportation          19712                   14969                    75.94
              June                 Department of Parks and Recreation          13787                    9164                    66.47
              June         

## 12. Resolution Quality by Month



In [14]:
# DATA QUALITY — RESOLUTION QUALITY BY MONTH

print("RESOLUTION QUALITY BY MONTH")
print("=" * 80)

monthly_resolution_quality = (
    df.groupby(
        ["created_month", "created_month_name", "resolution_quality"]
    )
    .size()
    .reset_index(name="record_count")
)

monthly_totals = (
    df.groupby(
        ["created_month", "created_month_name"]
    )
    .size()
    .reset_index(name="total_records")
)

monthly_resolution_quality = monthly_resolution_quality.merge(
    monthly_totals,
    on=["created_month", "created_month_name"],
    how="left"
)

monthly_resolution_quality["percentage"] = (
    monthly_resolution_quality["record_count"]
    / monthly_resolution_quality["total_records"]
    * 100
).round(2)

monthly_resolution_quality = monthly_resolution_quality.sort_values(
    ["created_month", "resolution_quality"]
)

print(
    monthly_resolution_quality[
        [
            "created_month_name",
            "resolution_quality",
            "record_count",
            "percentage"
        ]
    ].to_string(index=False)
)

print("\n" + "=" * 80)
print("MONTHLY RESOLUTION QUALITY SUMMARY")
print("=" * 80)

monthly_quality_summary = (
    df.groupby(["created_month", "created_month_name"])
      .agg(
          total_records=("service_request_id", "size"),
          valid_resolution=("resolution_hours", "count"),
          missing_invalid_closed=(
              "closed_at",
              lambda x: x.isna().sum()
          ),
          negative_duration=(
              "resolution_quality",
              lambda x: (x == "Negative Duration").sum()
          )
      )
      .reset_index()
)

monthly_quality_summary["valid_resolution_pct"] = (
    monthly_quality_summary["valid_resolution"]
    / monthly_quality_summary["total_records"]
    * 100
).round(2)

print(
    monthly_quality_summary[
        [
            "created_month_name",
            "total_records",
            "valid_resolution",
            "valid_resolution_pct",
            "missing_invalid_closed",
            "negative_duration"
        ]
    ].to_string(index=False)
)

print("\n" + "=" * 80)
print("RESOLUTION QUALITY BY MONTH COMPLETE")
print("=" * 80)


RESOLUTION QUALITY BY MONTH
created_month_name            resolution_quality  record_count  percentage
           January Missing / Invalid Closed Date          2603        0.75
           January             Negative Duration            99        0.03
           January                         Valid        345478       99.22
          February Missing / Invalid Closed Date          3551        1.39
          February             Negative Duration           129        0.05
          February                         Valid        251684       98.56
             March Missing / Invalid Closed Date          3959        1.41
             March             Negative Duration           136        0.05
             March                         Valid        277126       98.54
             April Missing / Invalid Closed Date          4914        1.80
             April             Negative Duration            91        0.03
             April                         Valid        267544       98.

## 13. Complaint Type Resolution Profile



In [15]:
# DATA PROFILING — COMPLAINT TYPE RESOLUTION PROFILE

print("COMPLAINT TYPE RESOLUTION PROFILE")
print("=" * 80)

complaint_resolution_profile = (
    df.dropna(subset=["complaint_type"])
      .groupby("complaint_type")
      .agg(
          request_count=("service_request_id", "size"),
          valid_resolution_count=("resolution_hours", "count"),
          median_resolution_hours=("resolution_hours", "median"),
          p75_resolution_hours=(
              "resolution_hours",
              lambda x: x.quantile(0.75)
          ),
          p90_resolution_hours=(
              "resolution_hours",
              lambda x: x.quantile(0.90)
          )
      )
      .reset_index()
)

complaint_resolution_profile["resolution_coverage_pct"] = (
    complaint_resolution_profile["valid_resolution_count"]
    / complaint_resolution_profile["request_count"]
    * 100
).round(2)

complaint_resolution_profile["median_resolution_days"] = (
    complaint_resolution_profile["median_resolution_hours"]
    / 24
).round(2)

# Keep complaint types with sufficient volume
complaint_resolution_profile_filtered = (
    complaint_resolution_profile[
        complaint_resolution_profile["request_count"] >= 1000
    ]
    .sort_values("request_count", ascending=False)
)

print(
    "Complaint types with at least 1,000 requests:",
    len(complaint_resolution_profile_filtered)
)

print("\nTOP 20 BY REQUEST VOLUME")
print("-" * 80)

print(
    complaint_resolution_profile_filtered.head(20)[
        [
            "complaint_type",
            "request_count",
            "resolution_coverage_pct",
            "median_resolution_hours",
            "median_resolution_days",
            "p75_resolution_hours",
            "p90_resolution_hours"
        ]
    ].to_string(index=False)
)

print("\n" + "=" * 80)
print("COMPLAINT TYPE RESOLUTION PROFILE COMPLETE")
print("=" * 80)


COMPLAINT TYPE RESOLUTION PROFILE
Complaint types with at least 1,000 requests: 122

TOP 20 BY REQUEST VOLUME
--------------------------------------------------------------------------------
          complaint_type  request_count  resolution_coverage_pct  median_resolution_hours  median_resolution_days  p75_resolution_hours  p90_resolution_hours
         Illegal Parking         564577                   100.00                 1.495000                    0.06              3.473889              7.089722
     Noise - Residential         456599                    99.99                 1.210000                    0.05              3.241667              8.128611
          HEAT/HOT WATER         315265                    99.51                39.357500                    1.64             57.039722             75.948722
        Blocked Driveway         169126                   100.00                 1.723056                    0.07              3.886042              7.817778
 Noise - Street/Sid

## 14. Longest-Median Complaint Types



In [16]:
# DATA PROFILING — LONGEST-MEDIAN COMPLAINT TYPES

print("LONGEST-MEDIAN COMPLAINT TYPES")
print("=" * 80)

long_resolution_complaints = (
    complaint_resolution_profile_filtered
    .sort_values(
        "median_resolution_hours",
        ascending=False
    )
)

print(
    long_resolution_complaints.head(20)[
        [
            "complaint_type",
            "request_count",
            "resolution_coverage_pct",
            "median_resolution_hours",
            "median_resolution_days",
            "p75_resolution_hours",
            "p90_resolution_hours"
        ]
    ].to_string(index=False)
)

print("\n" + "=" * 80)
print("LONGEST-MEDIAN COMPLAINT TYPES COMPLETE")
print("=" * 80)


LONGEST-MEDIAN COMPLAINT TYPES
                         complaint_type  request_count  resolution_coverage_pct  median_resolution_hours  median_resolution_days  p75_resolution_hours  p90_resolution_hours
                     Noise - Helicopter          20049                    52.85              7799.687222                  324.99           8423.078750           8976.197667
                         Taxi Complaint           9380                    74.09              3025.084167                  126.05           4874.577083           6374.511722
          Root/Sewer/Sidewalk Condition          10126                    72.46              2161.565278                   90.07           4540.777778           6353.167944
             For Hire Vehicle Complaint          17927                    85.22              2107.225278                   87.80           4183.760000           5868.015889
                           Building/Use          20087                   100.00              1920.927222

## 15. Common Resolution Durations



In [17]:
# DATA QUALITY — COMMON EXACT RESOLUTION DURATIONS

print("COMMON EXACT RESOLUTION DURATIONS")
print("=" * 80)

valid_resolution = df.loc[
    df["resolution_hours"].notna(),
    "resolution_hours"
]

duration_frequency = (
    valid_resolution
    .round(6)
    .value_counts()
    .reset_index()
)

duration_frequency.columns = [
    "resolution_hours",
    "record_count"
]

duration_frequency["resolution_days"] = (
    duration_frequency["resolution_hours"] / 24
).round(4)

duration_frequency["share_pct"] = (
    duration_frequency["record_count"]
    / len(valid_resolution)
    * 100
).round(3)

print(
    duration_frequency.head(30).to_string(index=False)
)

print("\n" + "=" * 80)
print("COMMON EXACT RESOLUTION DURATIONS COMPLETE")
print("=" * 80)


COMMON EXACT RESOLUTION DURATIONS
 resolution_hours  record_count  resolution_days  share_pct
         0.000000         63068           0.0000      1.789
         0.016667          1748           0.0007      0.050
         0.033333          1131           0.0014      0.032
         0.083333          1052           0.0035      0.030
         0.166667          1032           0.0069      0.029
         0.100000          1010           0.0042      0.029
         0.066667          1009           0.0028      0.029
         0.116667          1000           0.0049      0.028
         0.500000           983           0.0208      0.028
         0.050000           978           0.0021      0.028
         0.133333           928           0.0056      0.026
         0.150000           885           0.0062      0.025
         0.250000           847           0.0104      0.024
         0.416667           830           0.0174      0.024
         0.183333           815           0.0076      0.023
      

## 16. Zero-Duration Requests



In [18]:
# DATA QUALITY — ZERO-DURATION REQUESTS

print("ZERO-DURATION REQUEST ANALYSIS")
print("=" * 80)

zero_duration_df = df[
    df["resolution_hours"] == 0
].copy()

print(
    "Zero-duration records:",
    f"{len(zero_duration_df):,}"
)

print(
    "Share of valid resolution records:",
    f"{len(zero_duration_df) / df['resolution_hours'].notna().sum() * 100:.2f}%"
)

print("\nZERO-DURATION BY AGENCY")
print("-" * 80)

zero_by_agency = (
    zero_duration_df
    .groupby("agency")
    .size()
    .reset_index(name="zero_duration_count")
)

zero_by_agency["share_of_agency_zero_duration_pct"] = (
    zero_by_agency["zero_duration_count"]
    / len(zero_duration_df)
    * 100
).round(2)

zero_by_agency = zero_by_agency.sort_values(
    "zero_duration_count",
    ascending=False
)

print(
    zero_by_agency.to_string(index=False)
)

print("\n" + "=" * 80)
print("ZERO-DURATION REQUEST ANALYSIS COMPLETE")
print("=" * 80)


ZERO-DURATION REQUEST ANALYSIS
Zero-duration records: 63,068
Share of valid resolution records: 1.79%

ZERO-DURATION BY AGENCY
--------------------------------------------------------------------------------
                                            agency  zero_duration_count  share_of_agency_zero_duration_pct
           Department of Health and Mental Hygiene                24132                              38.26
            Department of Environmental Protection                17510                              27.76
                           Department of Buildings                12545                              19.89
                      Department of Transportation                 7985                              12.66
Department of Housing Preservation and Development                  848                               1.34
                          Department of Sanitation                   47                               0.07
                   Department of Homeless S

## 17. Final Profiling Summary



In [19]:
# FINAL DATA PROFILING & QUALITY SUMMARY

print("NYC 311 2025 — FINAL DATA PROFILING SUMMARY")
print("=" * 90)

total_records = len(df)

valid_resolution = df["resolution_hours"].notna().sum()
zero_duration = (df["resolution_hours"] == 0).sum()
negative_duration = (df["resolution_quality"] == "Negative Duration").sum()

valid_borough = df["borough"].notna().sum()
valid_board = df["community_board"].notna().sum()

valid_coordinates = (
    df["latitude"].notna()
    & df["longitude"].notna()
).sum()

valid_zip = df["incident_zip"].notna().sum()

primary_geo = (
    df["borough"].notna()
    | df["community_board"].notna()
    | (
        df["latitude"].notna()
        & df["longitude"].notna()
    )
).sum()

print(f"Total records:                  {total_records:,}")
print(f"Unique service request IDs:     {df['service_request_id'].nunique():,}")
print(f"Duplicate service request IDs:  {df['service_request_id'].duplicated().sum():,}")

print("\nDATE COVERAGE")
print("-" * 90)
print(f"Created date min:               {df['created_at'].min()}")
print(f"Created date max:               {df['created_at'].max()}")
print(f"Created dates missing:          {df['created_at'].isna().sum():,}")

print("\nRESOLUTION QUALITY")
print("-" * 90)
print(
    f"Valid resolution records:       {valid_resolution:,} "
    f"({valid_resolution / total_records * 100:.2f}%)"
)
print(
    f"Missing/invalid resolution:     "
    f"{total_records - valid_resolution - negative_duration:,}"
)
print(
    f"Negative duration records:      {negative_duration:,}"
)
print(
    f"Zero-duration records:          {zero_duration:,} "
    f"({zero_duration / valid_resolution * 100:.2f}% of valid)"
)

print("\nGEOGRAPHIC COVERAGE")
print("-" * 90)
print(
    f"Valid borough:                  {valid_borough:,} "
    f"({valid_borough / total_records * 100:.2f}%)"
)
print(
    f"Valid community board:           {valid_board:,} "
    f"({valid_board / total_records * 100:.2f}%)"
)
print(
    f"Valid coordinate pairs:          {valid_coordinates:,} "
    f"({valid_coordinates / total_records * 100:.2f}%)"
)
print(
    f"Valid ZIP codes:                 {valid_zip:,} "
    f"({valid_zip / total_records * 100:.2f}%)"
)
print(
    f"Primary geographic coverage:     {primary_geo:,} "
    f"({primary_geo / total_records * 100:.2f}%)"
)

print("\nCATEGORICAL COVERAGE")
print("-" * 90)
print(
    f"Agencies:                       {df['agency'].nunique():,}"
)
print(
    f"Complaint types:                {df['complaint_type'].nunique():,}"
)
print(
    f"Recognized statuses:            {df['status'].notna().sum():,}"
)

print("\nDATASET STRUCTURE")
print("-" * 90)
print(f"Rows:                            {df.shape[0]:,}")
print(f"Columns:                         {df.shape[1]:,}")

print("\n" + "=" * 90)
print("FINAL DATA PROFILING SUMMARY COMPLETE")
print("=" * 90)


NYC 311 2025 — FINAL DATA PROFILING SUMMARY
Total records:                  3,655,040
Unique service request IDs:     3,655,040
Duplicate service request IDs:  0

DATE COVERAGE
------------------------------------------------------------------------------------------
Created date min:               2025-01-01 00:00:12
Created date max:               2025-12-31 23:59:28
Created dates missing:          0

RESOLUTION QUALITY
------------------------------------------------------------------------------------------
Valid resolution records:       3,525,111 (96.45%)
Missing/invalid resolution:     129,036
Negative duration records:      893
Zero-duration records:          63,068 (1.79% of valid)

GEOGRAPHIC COVERAGE
------------------------------------------------------------------------------------------
Valid borough:                  3,396,220 (92.92%)
Valid community board:           3,246,595 (88.83%)
Valid coordinate pairs:          3,303,394 (90.38%)
Valid ZIP codes:                 

## Conclusion

The profiling stage establishes the major demand, resolution, geographic, and data-quality patterns used in the subsequent geospatial, socioeconomic, SQL, and Power BI analysis.
